Building from scratch

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Inception Block

In [26]:
class Inception(nn.Module):
  def __init__(self,in_channels,c1,c3_reduce,c3,c5_reduce,c5,pool_proj):
    super().__init__()
    #branch 1
    self.b1=nn.Conv2d(in_channels,c1,kernel_size=1)

    #branch 2
    self.b2=nn.Sequential(
        nn.Conv2d(in_channels=in_channels,out_channels=c3_reduce,kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=c3_reduce,out_channels=c3,kernel_size=3, padding=1)
    )
    #branch 3
    self.b3=nn.Sequential(
        nn.Conv2d(in_channels=in_channels,out_channels=c5_reduce,kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=c5_reduce,out_channels=c5,kernel_size=5, padding=2)
    )
    #branch 4
    self.b4=nn.Sequential(
        nn.MaxPool2d(kernel_size=3,stride=1,padding=1),
        nn.Conv2d(in_channels=in_channels,out_channels=pool_proj,kernel_size=1)
    )
  def forward(self,x):
    return torch.cat([
          F.relu(self.b1(x)),
          self.b2(x),
          self.b3(x),
          self.b4(x)],dim=1)

Auxiliary Classifier

In [27]:
class AuxClassifer(nn.Module):
  def __init__(self,in_channels,num_classes):
    super().__init__()
    self.avgpool=nn.AvgPool2d(kernel_size=5,stride=3)
    self.conv=nn.Conv2d(in_channels=in_channels,out_channels=128,kernel_size=1)
    self.fc1=nn.Linear(in_features=128,out_features=1024)
    self.fc2=nn.Linear(in_features=1024,out_features=num_classes)

    self.drop=nn.Dropout(0.7)

  def forward(self,x):
    x=self.avgpool(x)
    x=self.conv(x)

    x=x.reshape(x.size(0),-1)
    x=F.relu(self.fc1(x))
    x=self.drop(x)
    x=self.fc2(x)

    return x

Full GoogleNet

In [28]:
class GoogleNet(nn.Module):
  def __init__(self,num_classes=1000,aux_logits=True):
    super().__init__()
    self.aux_logits=aux_logits

    #initial layers
    self.conv1=nn.Conv2d(in_channels=3,out_channels=64,kernel_size=7,stride=2,padding=3)
    self.maxpool1=nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

    self.conv2=nn.Conv2d(in_channels=64,out_channels=64,kernel_size=1)
    self.conv3=nn.Conv2d(in_channels=64,out_channels=192,kernel_size=3,stride=2,padding=1)
    self.maxpool2=nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

    #inception blocks
    self.inception3a=Inception(in_channels=192,c1=64,c3_reduce=96,c3=128,c5_reduce=16,c5=32,pool_proj=32)
    self.inception3b=Inception(in_channels=256,c1=128,c3_reduce=128,c3=192,c5_reduce=32,c5=92,pool_proj=64)
    self.maxpool3=nn.MaxPool2d(kernel_size=3,padding=1,stride=2)

    self.inception4a=Inception(in_channels=476,c1=192,c3_reduce=96,c3=208,c5_reduce=16,c5=48,pool_proj=64)
    self.inception4b=Inception(in_channels=512,c1=160,c3_reduce=112,c3=224,c5_reduce=24,c5=64,pool_proj=64)
    self.inception4c=Inception(in_channels=512,c1=128,c3_reduce=128,c3=256,c5_reduce=24,c5=64,pool_proj=64)
    self.inception4d=Inception(in_channels=512,c1=112,c3_reduce=144,c3=288,c5_reduce=32,c5=64,pool_proj=64)
    self.inception4e=Inception(in_channels=528,c1=256,c3_reduce=160,c3=320,c5_reduce=32,c5=128,pool_proj=128)
    self.maxpool4=nn.MaxPool2d(kernel_size=3,stride=3,padding=1)

    self.inception5a=Inception(in_channels=832,c1=256,c3_reduce=160,c3=320,c5_reduce=32,c5=128,pool_proj=128)
    self.inception5b=Inception(in_channels=832,c1=384,c3_reduce=192,c3=384,c5_reduce=48,c5=128,pool_proj=128)

    #Auxiliary classifiers
    if aux_logits:
      self.aux1=AuxClassifer(in_channels=512,num_classes=num_classes)
      self.aux2=AuxClassifer(in_channels=528,num_classes=num_classes)

    #Final Layers
    self.avgpool=nn.AdaptiveAvgPool2d((1,1))
    self.dropout=nn.Dropout(0.4)
    self.fc=nn.Linear(in_features=1024,out_features=num_classes)

  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.maxpool1(x)

    x=F.relu(self.conv2(x))
    x=F.relu(self.conv3(x))
    x=self.maxpool2(x)

    #Inception 3
    x=self.inception3a(x)
    x=self.inception3b(x)
    x=self.maxpool3(x)

    #Inception 4
    x=self.inception4a(x)

    aux1=None
    if self.aux_logits and self.training:
      aux1=self.aux1(x)

    x=self.inception4b(x)
    x=self.inception4c(x)
    x=self.inception4d(x)

    aux2=None
    if self.aux_logits and self.training:
      aux2=self.aux2(x)

    x=self.inception4e(x)
    x=self.maxpool4(x)

    #Inception 5
    x=self.inception5a(x)
    x=self.inception5b(x)

    #Final
    x=self.avgpool(x)
    x=torch.flatten(x,1)
    x=self.dropout(x)
    x=self.fc(x)

    if self.aux_logits and self.training:
      return aux1,aux2,x

In [29]:
model = GoogleNet(num_classes=10)
x = torch.randn(1, 3, 224, 224)

model.train()
main_out, aux1, aux2 = model(x)

print(main_out.shape)  # (1, 10)
print(aux1.shape)
print(aux2.shape)

torch.Size([1, 10])
torch.Size([1, 10])
torch.Size([1, 10])


Transfer Learning

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

model=models.googlenet(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [31]:
import numpy as np
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

Importing Datasets

In [32]:
train_transform=transforms.Compose([
  transforms.Resize(256),
  transforms.RandomCrop(224),
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485,0.486,0.405],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root="./data",train=True,download=True,transform=train_transform)
test_data=datasets.CIFAR10(root="./data",train=False,download=True,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
test_loader=DataLoader(test_data,shuffle=False,batch_size=64,pin_memory=True,num_workers=2)

Changing only last layer

In [33]:
num_classes=10
model.fc=nn.Linear(1024,num_classes)

In [40]:
for param in model.parameters():
  param.requires_grad=False

# for finetuning
# for param in model.inception5b.parameters():
#     param.requires_grad = True

for param in model.fc.parameters():
  param.requires_grad=True

In [35]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [36]:
#use this only when u have unfreezed inception4a and 4d cuz they only consist aux classifier

# EPOCHS=1
# for epoch in range(EPOCHS):
#   model.train()
#   for images,labels in train_loader:
#     images,labels=images.to(device),labels.to(device)

#     optimizer.zero_grad()
#     outputs=model(images)
#     if isinstance(outputs,tuple):
#       main_out,aux1,aux2=outputs
#       loss1=criterion(main_out,labels)
#       loss2=criterion(aux1,labels)
#       loss3=criterion(aux2,labels)
#       loss=loss1+0.3*loss2+0.3*loss3
#     else:
#     loss=criterion(outputs,labels)

#     loss.backward()
#     optimizer.step()
#   print(f"Epoch: {epoch+1/EPOCHS} Loss per batch: {loss.item()/len(train_loader)}")

In [37]:
EPOCHS=1
for epoch in range(EPOCHS):
  model.train()
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)

    optimizer.zero_grad()
    outputs=model(images)

    # The torchvision GoogLeNet returns a named tuple (GoogLeNetOutputs) during training
    # when aux_logits are enabled (which they are by default for pretrained models).
    # We need to extract the main logits from this named tuple.
    if isinstance(outputs, tuple):
      main_outputs = outputs.logits
    else:
      # If the model does not return a tuple (e.g., during eval or if aux_logits were disabled),
      # outputs is already the main output.
      main_outputs = outputs

    loss=criterion(main_outputs,labels)

    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch+1/EPOCHS} Loss: {loss.item()}")

Epoch: 1.0 Loss: 0.8330033421516418


In [38]:
def evaluate(model,loader,device):
  model.eval()
  total=0
  correct=0
  with torch.no_grad():
   for images,labels in loader:
    images,labels=images.to(device),labels.to(device)
    output=model(images)

    _,predicted=torch.max(output,1)
    total +=labels.size(0)
    correct +=(predicted==labels).sum().item()

  accuracy=(correct/total)*100
  return accuracy

In [39]:
print(f"The accuracy of the model is {evaluate(model,test_loader,device)}%")

The accuracy of the model is 73.42%
